In [1]:
from pathlib import Path
from collections import defaultdict

def analyze_dataset(folder_path):
    """
    Analyze your current dataset structure
    """
    path = Path(folder_path)
    
    # Statistics
    stats = {
        'total_files': 0,
        'color_images': [],
        'masks_by_class': defaultdict(list),
        'other_files': []
    }
    
    for file in path.glob("*.tif"):
        stats['total_files'] += 1
        filename = file.name
        
        if "_Bin_" in filename:
            # Extract class name
            class_name = filename.split("_Bin_")[1].replace(".tif", "")
            stats['masks_by_class'][class_name].append(filename)
        elif "_Col.tif" in filename:
            stats['color_images'].append(filename)
        else:
            stats['other_files'].append(filename)
    
    # Print report
    print("="*60)
    print("DATASET ANALYSIS")
    print("="*60)
    print(f"Total files: {stats['total_files']}")
    print(f"\nColor images (_Col.tif): {len(stats['color_images'])}")
    
    print(f"\nMask classes found:")
    for class_name, files in sorted(stats['masks_by_class'].items()):
        print(f"  - {class_name}: {len(files)} masks")
    
    print(f"\nOther files (ignored): {len(stats['other_files'])}")
    if stats['other_files'][:5]:
        print("  Examples:", stats['other_files'][:5])
    
    print("="*60)
    
    return stats

# Usage
folder = r"D:\U2_verginie\output scv2"
stats = analyze_dataset(folder)

DATASET ANALYSIS
Total files: 2394

Color images (_Col.tif): 222

Mask classes found:
  - BlackRot: 157 masks
  - Knot: 116 masks
  - Stain: 224 masks
  - StainMinor: 48 masks
  - Unplaned: 23 masks
  - YellowRot: 50 masks

Other files (ignored): 1554
  Examples: ['2-18-25_4.34.47.402_Bot_3DRaw.tif', '2-18-25_4.34.47.402_Bot_Depth.tif', '2-18-25_4.34.47.402_Bot_Elevation.tif', '2-18-25_4.34.47.402_Bot_Lum.tif', '2-18-25_4.34.47.402_Bot_Scatter.tif']


In [4]:
import shutil
from pathlib import Path

def reorganize_dataset(source_folder, output_folder):
    """
    Organize data into proper structure:
    output/
      ├── images/
      └── masks/
          ├── Knot/
          ├── Stain/
          └── ...
    """
    source_path = Path(source_folder)
    output_path = Path(output_folder)
    
    # Create directories
    images_dir = output_path / "images"
    masks_dir = output_path / "masks"
    images_dir.mkdir(parents=True, exist_ok=True)
    masks_dir.mkdir(parents=True, exist_ok=True)
    
    stats = {'images': 0, 'masks': defaultdict(int)}
    
    # Process files
    for file in source_path.glob("*.tif"):
        filename = file.name
        
        # Copy masks
        if "_Bin_" in filename:
            class_name = filename.split("_Bin_")[1].replace(".tif", "")
            class_dir = masks_dir / class_name
            class_dir.mkdir(exist_ok=True)
            
            shutil.copy2(file, class_dir / filename)
            stats['masks'][class_name] += 1
            
        # Copy images
        elif "_Col.tif" in filename:
            shutil.copy2(file, images_dir / filename)
            stats['images'] += 1
    
    # Report
    print("="*60)
    print("REORGANIZATION COMPLETE")
    print("="*60)
    print(f"Images copied: {stats['images']}")
    print(f"\nMasks by class:")
    for class_name, count in sorted(stats['masks'].items()):
        print(f"  {class_name}: {count}")
    print("="*60)
    
    return stats

# Usage
source = r"D:\U2_verginie\output scv2"
output = "path/to/organized/folder"
reorganize_dataset(source, output)

REORGANIZATION COMPLETE
Images copied: 222

Masks by class:
  BlackRot: 157
  Knot: 116
  Stain: 224
  StainMinor: 48
  Unplaned: 23
  YellowRot: 50


{'images': 222,
 'masks': defaultdict(int,
             {'Knot': 116,
              'Stain': 224,
              'StainMinor': 48,
              'BlackRot': 157,
              'YellowRot': 50,
              'Unplaned': 23})}

In [7]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp311-cp311-win_amd64.whl.metadata (27 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp311-cp311-win_amd64.whl.metadata (6.8 kB)
     ---------------------------------------- 0.0/536.2 kB ? eta -:--:--
     ---------------------------------------- 536.2/536.2 kB 8.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 67.3 MB/s eta 0:00:42
   ---------------------------------------- 0.0/2.8 GB 73.0 MB/s eta 0:00:39
    --------------------------------------- 0.0/2.8 GB 75.1 MB/s eta 0:00:37
    --------------------------------------- 0.1/2.8 GB 74.7 MB/s eta 0:00:37
   - -------------------------------------- 0.1

In [8]:
!pip install segmentation-models-pytorch albumentations opencv-python pillow numpy scikit-learn tqdm matplotlib tensorboard

  Using cached segmentation_models_pytorch-0.5.0-py3-none-any.whl.metadata (17 kB)
  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached matplotlib-3.10.6-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl.metadata (4.1 kB)
  Using cached pydantic-2.11.9-py3-none-any.whl.metadata (68 kB)
  Using cached albucore-0.0.24-py3-none-any.whl.metadata (5.3 kB)
  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.60.1-cp311-cp311-win_amd64.whl.

In [1]:
import numpy as np
from PIL import Image
from pathlib import Path

def validate_dataset(organized_folder):
    """
    Check that each image has corresponding masks
    and validate mask properties
    """
    base_path = Path(organized_folder)
    images_dir = base_path / "images"
    masks_dir = base_path / "masks"
    
    # Get all images
    image_files = sorted(images_dir.glob("*.tif"))
    
    # Get all mask classes
    mask_classes = [d.name for d in masks_dir.iterdir() if d.is_dir()]
    
    print("="*60)
    print("DATASET VALIDATION")
    print("="*60)
    print(f"Total images: {len(image_files)}")
    print(f"Mask classes: {mask_classes}")
    print()
    
    issues = []
    valid_samples = []
    
    for img_file in image_files:
        # Extract base name
        base_name = img_file.stem.replace("_Col", "")
        
        # Find corresponding masks
        masks_found = {}
        for class_name in mask_classes:
            mask_pattern = f"*{base_name}*_Bin_{class_name}.tif"
            mask_files = list((masks_dir / class_name).glob(mask_pattern))
            
            if mask_files:
                masks_found[class_name] = mask_files[0]
        
        # Check if at least one mask exists
        if not masks_found:
            issues.append(f"No masks for: {img_file.name}")
            continue
        
        # Validate dimensions
        img = Image.open(img_file)
        img_size = img.size
        
        for class_name, mask_file in masks_found.items():
            mask = Image.open(mask_file)
            if mask.size != img_size:
                issues.append(f"Size mismatch: {img_file.name} vs {mask_file.name}")
        
        valid_samples.append({
            'image': img_file,
            'masks': masks_found,
            'size': img_size
        })
    
    # Print results
    print(f"Valid samples: {len(valid_samples)}")
    print(f"Issues found: {len(issues)}")
    
    if issues:
        print("\nIssues:")
        for issue in issues[:10]:
            print(f"  - {issue}")
        if len(issues) > 10:
            print(f"  ... and {len(issues) - 10} more")
    
    print("="*60)
    
    return valid_samples, issues

# Usage
organized_folder = "path/to/organized/folder"
valid_samples, issues = validate_dataset(organized_folder)

DATASET VALIDATION
Total images: 222
Mask classes: ['BlackRot', 'Knot', 'Stain', 'StainMinor', 'Unplaned', 'YellowRot']

Valid samples: 222
Issues found: 0


In [2]:
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm

def create_combined_masks(organized_folder):
    """
    Combine multiple binary masks into single multi-class mask
    Background = 0, Class1 = 1, Class2 = 2, etc.
    """
    base_path = Path(organized_folder)
    images_dir = base_path / "images"
    masks_dir = base_path / "masks"
    combined_masks_dir = base_path / "combined_masks"
    combined_masks_dir.mkdir(exist_ok=True)
    
    # Get mask classes
    mask_classes = sorted([d.name for d in masks_dir.iterdir() if d.is_dir()])
    class_mapping = {class_name: idx + 1 for idx, class_name in enumerate(mask_classes)}
    
    print("="*60)
    print("CREATING COMBINED MASKS")
    print("="*60)
    print("Class mapping:")
    print("  0: Background")
    for class_name, idx in class_mapping.items():
        print(f"  {idx}: {class_name}")
    print()
    
    # Process each image
    image_files = sorted(images_dir.glob("*.tif"))
    
    for img_file in tqdm(image_files, desc="Processing"):
        base_name = img_file.stem.replace("_Col", "")
        
        # Load image to get dimensions
        img = Image.open(img_file)
        width, height = img.size
        
        # Create empty combined mask
        combined_mask = np.zeros((height, width), dtype=np.uint8)
        
        # Overlay each class mask
        for class_name, class_idx in class_mapping.items():
            mask_pattern = f"*{base_name}*_Bin_{class_name}.tif"
            mask_files = list((masks_dir / class_name).glob(mask_pattern))
            
            if mask_files:
                mask = np.array(Image.open(mask_files[0]))
                
                # Handle RGB masks (take first channel)
                if len(mask.shape) == 3:
                    mask = mask[:, :, 0]
                
                # Set pixels to class index where mask is active
                combined_mask[mask > 127] = class_idx
        
        # Save combined mask
        output_file = combined_masks_dir / f"{base_name}_mask.png"
        Image.fromarray(combined_mask).save(output_file)
    
    print(f"\nCombined masks saved to: {combined_masks_dir}")
    print("="*60)
    
    return class_mapping

# Usage
organized_folder = "path/to/organized/folder"
class_mapping = create_combined_masks(organized_folder)

CREATING COMBINED MASKS
Class mapping:
  0: Background
  1: BlackRot
  2: Knot
  3: Stain
  4: StainMinor
  5: Unplaned
  6: YellowRot



Processing: 100%|██████████| 222/222 [00:34<00:00,  6.47it/s]


Combined masks saved to: path\to\organized\folder\combined_masks


In [1]:
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

def create_train_val_split(organized_folder, val_split=0.2, seed=42):
    """
    Split dataset into train and validation sets
    """
    base_path = Path(organized_folder)
    images_dir = base_path / "images"
    combined_masks_dir = base_path / "combined_masks"
    
    # Get all image files
    image_files = sorted([f.stem.replace("_Col", "") for f in images_dir.glob("*.tif")])
    
    # Split
    train_ids, val_ids = train_test_split(
        image_files, 
        test_size=val_split, 
        random_state=seed
    )
    
    # Save split info
    split_info = {
        'train': train_ids,
        'val': val_ids,
        'num_train': len(train_ids),
        'num_val': len(val_ids)
    }
    
    split_file = base_path / "split.json"
    with open(split_file, 'w') as f:
        json.dump(split_info, f, indent=2)
    
    print("="*60)
    print("TRAIN/VAL SPLIT")
    print("="*60)
    print(f"Train samples: {len(train_ids)}")
    print(f"Val samples: {len(val_ids)}")
    print(f"Split saved to: {split_file}")
    print("="*60)
    
    return split_info

# Usage
organized_folder = "path/to/organized/folder"
split_info = create_train_val_split(organized_folder, val_split=0.2)

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [4]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import json
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2

class SegmentationDataset(Dataset):
    """
    PyTorch Dataset for segmentation
    """
    def __init__(self, data_root, split='train', transform=None):
        """
        Args:
            data_root: Path to organized folder
            split: 'train' or 'val'
            transform: Albumentations transform
        """
        self.data_root = Path(data_root)
        self.split = split
        self.transform = transform
        
        # Load split info
        split_file = self.data_root / "split.json"
        with open(split_file, 'r') as f:
            split_info = json.load(f)
        
        self.image_ids = split_info[split]
        
        # Paths
        self.images_dir = self.data_root / "images"
        self.masks_dir = self.data_root / "combined_masks"
        
        print(f"Loaded {split} dataset: {len(self.image_ids)} samples")
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        # Get image ID
        img_id = self.image_ids[idx]
        
        # Load image
        img_path = self.images_dir / f"{img_id}_Col.tif"
        image = np.array(Image.open(img_path).convert('RGB'))
        
        # Load mask
        mask_path = self.masks_dir / f"{img_id}_mask.png"
        mask = np.array(Image.open(mask_path))
        
        # Apply transforms
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        
        return image, mask.long()


def get_training_augmentation(img_size=512):
    """
    Training augmentations
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.1, 
            scale_limit=0.1, 
            rotate_limit=15, 
            p=0.5
        ),
        A.OneOf([
            A.RandomBrightnessContrast(p=1),
            A.RandomGamma(p=1),
        ], p=0.3),
        A.OneOf([
            A.GaussNoise(p=1),
            A.GaussianBlur(p=1),
        ], p=0.2),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2()
    ])


def get_validation_augmentation(img_size=512):
    """
    Validation augmentations (resize + normalize only)
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2()
    ])


def create_dataloaders(data_root, batch_size=4, img_size=512, num_workers=4):
    """
    Create train and validation dataloaders
    """
    # Create datasets
    train_dataset = SegmentationDataset(
        data_root=data_root,
        split='train',
        transform=get_training_augmentation(img_size)
    )
    
    val_dataset = SegmentationDataset(
        data_root=data_root,
        split='val',
        transform=get_validation_augmentation(img_size)
    )
    
    # Create dataloaders
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )
    
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )
    
    print("="*60)
    print("DATALOADERS CREATED")
    print("="*60)
    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
    print(f"Batch size: {batch_size}")
    print(f"Image size: {img_size}x{img_size}")
    print("="*60)
    
    return train_loader, val_loader


# Test the dataset
if __name__ == "__main__":
    data_root = "path/to/organized/folder"
    
    # Create dataloaders
    train_loader, val_loader = create_dataloaders(
        data_root=data_root,
        batch_size=4,
        img_size=512,
        num_workers=0  # Set to 0 for debugging on Windows
    )
    
    # Test loading a batch
    print("\nTesting batch loading...")
    images, masks = next(iter(train_loader))
    print(f"Images shape: {images.shape}")
    print(f"Masks shape: {masks.shape}")
    print(f"Mask unique values: {torch.unique(masks)}")
    print("\nDataset test successful!")

d:\U2_verginie\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\U2_verginie\.venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
d:\U2_verginie\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Loaded train dataset: 177 samples
Loaded val dataset: 45 samples
DATALOADERS CREATED
Train batches: 45
Val batches: 12
Batch size: 4
Image size: 512x512

Testing batch loading...
Images shape: torch.Size([4, 3, 512, 512])
Masks shape: torch.Size([4, 512, 512])
Mask unique values: tensor([0, 1, 2, 3, 4, 5])

Dataset test successful!


In [5]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from pathlib import Path
import time
from tqdm import tqdm
import numpy as np

class SegmentationTrainer:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        num_classes,
        device='cuda',
        lr=1e-4,
        save_dir='checkpoints'
    ):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.num_classes = num_classes
        self.device = device
        
        # Loss and optimizer
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        # Scheduler
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', patience=5, factor=0.5
        )
        
        # Metrics tracking
        self.train_losses = []
        self.val_losses = []
        self.best_val_loss = float('inf')
        
        # Save directory
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(exist_ok=True)
    
    def calculate_iou(self, pred, target, num_classes):
        """Calculate IoU for each class"""
        ious = []
        pred = pred.view(-1)
        target = target.view(-1)
        
        for cls in range(num_classes):
            pred_cls = (pred == cls)
            target_cls = (target == cls)
            
            intersection = (pred_cls & target_cls).sum().float()
            union = (pred_cls | target_cls).sum().float()
            
            if union == 0:
                ious.append(float('nan'))
            else:
                ious.append((intersection / union).item())
        
        return ious
    
    def train_epoch(self, epoch):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch} [TRAIN]')
        for images, masks in pbar:
            images = images.to(self.device)
            masks = masks.to(self.device)
            
            # Forward
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, masks)
            
            # Backward
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(self.train_loader)
        self.train_losses.append(avg_loss)
        return avg_loss
    
    def validate(self, epoch):
        """Validate the model"""
        self.model.eval()
        total_loss = 0
        all_ious = []
        
        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc=f'Epoch {epoch} [VAL]')
            for images, masks in pbar:
                images = images.to(self.device)
                masks = masks.to(self.device)
                
                # Forward
                outputs = self.model(images)
                loss = self.criterion(outputs, masks)
                total_loss += loss.item()
                
                # Calculate IoU
                preds = torch.argmax(outputs, dim=1)
                ious = self.calculate_iou(preds, masks, self.num_classes)
                all_ious.append(ious)
                
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(self.val_loader)
        self.val_losses.append(avg_loss)
        
        # Calculate mean IoU
        mean_ious = np.nanmean(all_ious, axis=0)
        
        return avg_loss, mean_ious
    
    def save_checkpoint(self, epoch, val_loss, is_best=False):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'val_loss': val_loss,
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
        }
        
        # Save latest
        torch.save(checkpoint, self.save_dir / 'latest.pth')
        
        # Save best
        if is_best:
            torch.save(checkpoint, self.save_dir / 'best.pth')
            print(f"✓ Saved best model (val_loss: {val_loss:.4f})")
    
    def train(self, num_epochs):
        """Full training loop"""
        print("="*60)
        print("STARTING TRAINING")
        print("="*60)
        print(f"Device: {self.device}")
        print(f"Epochs: {num_epochs}")
        print(f"Num classes: {self.num_classes}")
        print("="*60)
        
        for epoch in range(1, num_epochs + 1):
            # Train
            train_loss = self.train_epoch(epoch)
            
            # Validate
            val_loss, mean_ious = self.validate(epoch)
            
            # Learning rate scheduling
            self.scheduler.step(val_loss)
            current_lr = self.optimizer.param_groups[0]['lr']
            
            # Print results
            print(f"\nEpoch {epoch}/{num_epochs}:")
            print(f"  Train Loss: {train_loss:.4f}")
            print(f"  Val Loss:   {val_loss:.4f}")
            print(f"  Learning Rate: {current_lr:.6f}")
            print(f"  Mean IoU per class:")
            for cls, iou in enumerate(mean_ious):
                if not np.isnan(iou):
                    print(f"    Class {cls}: {iou:.4f}")
            print("-"*60)
            
            # Save checkpoint
            is_best = val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss
            
            self.save_checkpoint(epoch, val_loss, is_best)
        
        print("="*60)
        print("TRAINING COMPLETE")
        print(f"Best Val Loss: {self.best_val_loss:.4f}")
        print("="*60)


def create_model(num_classes, encoder_name='resnet34', pretrained=True):
    """
    Create segmentation model
    
    Popular encoders:
    - 'resnet34', 'resnet50' (good balance)
    - 'efficientnet-b0' to 'efficientnet-b7' (efficient)
    - 'timm-efficientnet-b5' (powerful)
    """
    model = smp.Unet(
        encoder_name=encoder_name,
        encoder_weights='imagenet' if pretrained else None,
        in_channels=3,
        classes=num_classes,
    )
    
    print("="*60)
    print("MODEL CREATED")
    print("="*60)
    print(f"Architecture: U-Net")
    print(f"Encoder: {encoder_name}")
    print(f"Pretrained: {pretrained}")
    print(f"Classes: {num_classes}")
    print("="*60)
    
    return model


# Main training script
if __name__ == "__main__":
    from step6_dataset import create_dataloaders
    
    # Configuration
    DATA_ROOT = "path/to/organized/folder"
    NUM_CLASSES = 6  # 0=background + 5 defect classes
    BATCH_SIZE = 4
    IMG_SIZE = 512
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-4
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Create dataloaders
    train_loader, val_loader = create_dataloaders(
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE,
        num_workers=0  # Windows: use 0, Linux: use 4
    )
    
    # Create model
    model = create_model(
        num_classes=NUM_CLASSES,
        encoder_name='resnet34',
        pretrained=True
    )
    
    # Create trainer
    trainer = SegmentationTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_classes=NUM_CLASSES,
        device=DEVICE,
        lr=LEARNING_RATE,
        save_dir='checkpoints'
    )
    
    # Start training
    trainer.train(num_epochs=NUM_EPOCHS)

ModuleNotFoundError: No module named 'step6_dataset'

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2


# ============================================================
# DATASET
# ============================================================

class SegmentationDataset(Dataset):
    def __init__(self, data_root, split='train', transform=None):
        self.data_root = Path(data_root)
        self.split = split
        self.transform = transform
        
        # Load split info
        split_file = self.data_root / "split.json"
        with open(split_file, 'r') as f:
            split_info = json.load(f)
        
        self.image_ids = split_info[split]
        self.images_dir = self.data_root / "images"
        self.masks_dir = self.data_root / "combined_masks"
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        
        # Load image
        img_path = self.images_dir / f"{img_id}_Col.tif"
        image = np.array(Image.open(img_path).convert('RGB'))
        
        # Load mask
        mask_path = self.masks_dir / f"{img_id}_mask.png"
        mask = np.array(Image.open(mask_path))
        
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        
        return image, mask.long()


def get_training_augmentation(img_size=512):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.1, 
            scale_limit=0.1, 
            rotate_limit=15, 
            p=0.5
        ),
        A.OneOf([
            A.RandomBrightnessContrast(p=1),
            A.RandomGamma(p=1),
        ], p=0.3),
        A.OneOf([
            A.GaussNoise(p=1),
            A.GaussianBlur(p=1),
        ], p=0.2),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2()
    ])


def get_validation_augmentation(img_size=512):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2()
    ])


def get_num_classes(data_root):
    """Automatically detect number of classes from masks"""
    masks_dir = Path(data_root) / "combined_masks"
    all_classes = set()
    
    for mask_file in list(masks_dir.glob("*.png"))[:10]:  # Check first 10 masks
        mask = np.array(Image.open(mask_file))
        all_classes.update(np.unique(mask).tolist())
    
    num_classes = len(all_classes)
    print(f"Detected {num_classes} classes: {sorted(all_classes)}")
    return num_classes


def create_dataloaders(data_root, batch_size=4, img_size=512, num_workers=0):
    train_dataset = SegmentationDataset(
        data_root=data_root,
        split='train',
        transform=get_training_augmentation(img_size)
    )
    
    val_dataset = SegmentationDataset(
        data_root=data_root,
        split='val',
        transform=get_validation_augmentation(img_size)
    )
    
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )
    
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )
    
    print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
    print(f"Val: {len(val_dataset)} samples, {len(val_loader)} batches")
    
    return train_loader, val_loader


# ============================================================
# MODEL
# ============================================================

def create_model(num_classes, encoder_name='resnet34', pretrained=True):
    model = smp.Unet(
        encoder_name=encoder_name,
        encoder_weights='imagenet' if pretrained else None,
        in_channels=3,
        classes=num_classes,
    )
    
    print("="*60)
    print("MODEL CREATED")
    print("="*60)
    print(f"Architecture: U-Net")
    print(f"Encoder: {encoder_name}")
    print(f"Pretrained: {pretrained}")
    print(f"Classes: {num_classes}")
    print("="*60)
    
    return model


# ============================================================
# TRAINER
# ============================================================

class SegmentationTrainer:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        num_classes,
        device='cuda',
        lr=1e-4,
        save_dir='checkpoints'
    ):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.num_classes = num_classes
        self.device = device
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', patience=5, factor=0.5
        )
        
        self.train_losses = []
        self.val_losses = []
        self.best_val_loss = float('inf')
        
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(exist_ok=True)
    
    def calculate_iou(self, pred, target, num_classes):
        ious = []
        pred = pred.view(-1)
        target = target.view(-1)
        
        for cls in range(num_classes):
            pred_cls = (pred == cls)
            target_cls = (target == cls)
            
            intersection = (pred_cls & target_cls).sum().float()
            union = (pred_cls | target_cls).sum().float()
            
            if union == 0:
                ious.append(float('nan'))
            else:
                ious.append((intersection / union).item())
        
        return ious
    
    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch} [TRAIN]')
        for images, masks in pbar:
            images = images.to(self.device)
            masks = masks.to(self.device)
            
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, masks)
            
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(self.train_loader)
        self.train_losses.append(avg_loss)
        return avg_loss
    
    def validate(self, epoch):
        self.model.eval()
        total_loss = 0
        all_ious = []
        
        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc=f'Epoch {epoch} [VAL]')
            for images, masks in pbar:
                images = images.to(self.device)
                masks = masks.to(self.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, masks)
                total_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1)
                ious = self.calculate_iou(preds, masks, self.num_classes)
                all_ious.append(ious)
                
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(self.val_loader)
        self.val_losses.append(avg_loss)
        mean_ious = np.nanmean(all_ious, axis=0)
        
        return avg_loss, mean_ious
    
    def save_checkpoint(self, epoch, val_loss, is_best=False):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'val_loss': val_loss,
        }
        
        torch.save(checkpoint, self.save_dir / 'latest.pth')
        
        if is_best:
            torch.save(checkpoint, self.save_dir / 'best.pth')
            print(f"✓ Saved best model (val_loss: {val_loss:.4f})")
    
    def train(self, num_epochs):
        print("="*60)
        print("STARTING TRAINING")
        print("="*60)
        print(f"Device: {self.device}")
        print(f"Epochs: {num_epochs}")
        print(f"Num classes: {self.num_classes}")
        print("="*60)
        
        for epoch in range(1, num_epochs + 1):
            train_loss = self.train_epoch(epoch)
            val_loss, mean_ious = self.validate(epoch)
            
            self.scheduler.step(val_loss)
            current_lr = self.optimizer.param_groups[0]['lr']
            
            print(f"\nEpoch {epoch}/{num_epochs}:")
            print(f"  Train Loss: {train_loss:.4f}")
            print(f"  Val Loss:   {val_loss:.4f}")
            print(f"  Learning Rate: {current_lr:.6f}")
            print(f"  Mean IoU per class:")
            for cls, iou in enumerate(mean_ious):
                if not np.isnan(iou):
                    print(f"    Class {cls}: {iou:.4f}")
            print("-"*60)
            
            is_best = val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss
            
            self.save_checkpoint(epoch, val_loss, is_best)
        
        print("="*60)
        print("TRAINING COMPLETE")
        print(f"Best Val Loss: {self.best_val_loss:.4f}")
        print("="*60)


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    # Configuration - UPDATE THIS PATH
    DATA_ROOT = r"D:\U2_verginie\path\to\organized\folder"
    
    BATCH_SIZE = 4
    IMG_SIZE = 512
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-4
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    print("="*60)
    print("CONFIGURATION")
    print("="*60)
    print(f"Data root: {DATA_ROOT}")
    print(f"Device: {DEVICE}")
    print("="*60)
    
    # Auto-detect number of classes
    NUM_CLASSES = get_num_classes(DATA_ROOT)
    
    # Create dataloaders
    train_loader, val_loader = create_dataloaders(
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE,
        num_workers=0
    )
    
    # Create model
    model = create_model(
        num_classes= 7,
        encoder_name='resnet34',
        pretrained=True
    )
    
    # Train
    trainer = SegmentationTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_classes=NUM_CLASSES,
        device=DEVICE,
        lr=LEARNING_RATE,
        save_dir='checkpoints'
    )
    
    trainer.train(num_epochs=NUM_EPOCHS)

CONFIGURATION
Data root: D:\U2_verginie\path\to\organized\folder
Device: cpu
Detected 5 classes: [0, 1, 2, 3, 4]
Train: 177 samples, 45 batches
Val: 45 samples, 12 batches


'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'Une connexion existante a dû être fermée par l’hôte distant', None, 10054, None)), '(Request ID: 2d70ef03-9139-49e3-b04b-3c5647f3e614)')' thrown while requesting HEAD https://huggingface.co/smp-hub/resnet34.imagenet/resolve/7a57b34f723329ff020b3f8bc41771163c519d0c/config.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'Une connexion existante a dû être fermée par l’hôte distant', None, 10054, None)), '(Request ID: 8e02d8d2-701a-4327-bb89-2f47837156c7)')' thrown while requesting HEAD https://huggingface.co/smp-hub/resnet34.imagenet/resolve/7a57b34f723329ff020b3f8bc41771163c519d0c/config.json
Retrying in 2s [Retry 2/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'Une connexion existante a dû être fermée par l’hôte distant', None, 10054, None)), '(Request ID: 1dadb54f-9a2e-4e84-8f23-5a2035753f60)')' thrown while requesting HEAD https://huggingface

MODEL CREATED
Architecture: U-Net
Encoder: resnet34
Pretrained: True
Classes: 7
STARTING TRAINING
Device: cpu
Epochs: 50
Num classes: 5


Epoch 1 [VAL]: 100%|██████████| 12/12 [00:23<00:00,  1.97s/it, loss=1.5421]



Epoch 1/50:
  Train Loss: 1.5799
  Val Loss:   1.5101
  Learning Rate: 0.000100
  Mean IoU per class:
    Class 0: 0.4879
    Class 1: 0.0025
    Class 2: 0.0014
    Class 3: 0.1194
    Class 4: 0.0024
------------------------------------------------------------
✓ Saved best model (val_loss: 1.5101)


Epoch 2 [VAL]: 100%|██████████| 12/12 [00:20<00:00,  1.71s/it, loss=1.7417]



Epoch 2/50:
  Train Loss: 1.3112
  Val Loss:   1.3651
  Learning Rate: 0.000100
  Mean IoU per class:
    Class 0: 0.5769
    Class 1: 0.0008
    Class 2: 0.0001
    Class 3: 0.2326
    Class 4: 0.0012
------------------------------------------------------------
✓ Saved best model (val_loss: 1.3651)


Epoch 3 [VAL]: 100%|██████████| 12/12 [00:23<00:00,  1.97s/it, loss=1.8963]



Epoch 3/50:
  Train Loss: 1.1946
  Val Loss:   1.3149
  Learning Rate: 0.000100
  Mean IoU per class:
    Class 0: 0.5658
    Class 1: 0.0001
    Class 2: 0.0000
    Class 3: 0.1904
    Class 4: 0.0000
------------------------------------------------------------
✓ Saved best model (val_loss: 1.3149)


Epoch 4 [TRAIN]:  13%|█▎        | 6/45 [00:34<03:41,  5.68s/it, loss=1.1413]